In [2]:
import tensorflow as tf
from pathlib import Path

In [ ]:
tf.random.set_seed(123)

project_dir = Path.cwd().parent
train_dir = project_dir / "img/train"
validation_dir = project_dir / "img/validate"

img_height = 180
img_width = 180
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

""" 
I created another notebook to give EfficientNetB0 architecture a try but there seems 
to be an issue with my versions of tf and keras as I tried many different tweaks and 
kept getting a shape mismatch error, so the MobileNetV2 architecture below seems 
like the best fit for my small dataset if I want to use transfer learning
"""
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

model.save('../models/mobilenetv2_model.keras')

""" 
import numpy as np 

train = np.array([80,84,78,80,80,83,66,81,83,82,81,83,82,87,76,85,80,86,76,88,76,84,83,85,72])
train_avg = train.mean()
validate = np.array([88,93,88,93,90,90,90,83,90,86,90,88,86,88,86,83,88,88,79,83,86,88,90,83,86])
validate_avg = validate.mean()

print(f"train average: {train_avg:.2f}")
print(f"validate average: {validate_avg:.2f}")
print(f"Training runs: {len(train)}")
"""

Found 161 files belonging to 3 classes.
Found 42 files belonging to 3 classes.


/var/folders/04/s_rk8gfn6vq1608hxwwtl_040000gn/T/ipykernel_25112/2627652257.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.4161 - loss: 2.6194 - val_accuracy: 0.5714 - val_loss: 2.2096
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 177ms/step - accuracy: 0.4720 - loss: 1.5923 - val_accuracy: 0.5238 - val_loss: 1.3801
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - accuracy: 0.5217 - loss: 1.7792 - val_accuracy: 0.7619 - val_loss: 0.4683
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step - accuracy: 0.6273 - loss: 1.1755 - val_accuracy: 0.7857 - val_loss: 0.5183
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.6522 - loss: 1.0230 - val_accuracy: 0.7619 - val_loss: 0.4905
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.6584 - loss: 0.9936 - val_accuracy: 0.8810 - val_loss: 0.3600
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.6335 - loss: 1.0911 - val_accuracy: 0.7619 - val_loss: 0.4815
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.6894 - loss: 0.9691 - val_accuracy: 0.8333 - val_loss: 0.3